## Imports & setup

In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker()
Faker.seed(SEED)


In [3]:
# Simulation config

CONFIG = {
    "n_legit_users"    : 800,
    "n_bot_users"      : 200,
    "n_videos"         : 500,
    "days_window"      : 30,
    "events_per_legit" : (5, 50),
    "events_per_bot"   : (200, 500),
    "bot_ip_pool"      : 20,
}

TOTAL_USERS = CONFIG["n_legit_users"] + CONFIG["n_bot_users"]

print(f"Total users    : {TOTAL_USERS}")
print(f"Bot ratio      : {CONFIG['n_bot_users'] / TOTAL_USERS:.1%}")
print(f"Simulation days: {CONFIG['days_window']}")

Total users    : 1000
Bot ratio      : 20.0%
Simulation days: 30


In [4]:
def generate_users(config):
    users = []
    start_date = datetime(2024, 1, 1)
    bot_ip_pool = [fake.ipv4_public() for _ in range(config["bot_ip_pool"])]

    # Legitimate users
    for i in range(config["n_legit_users"]):
        created = start_date - timedelta(days=random.randint(30, 730))
        users.append({
            "user_id"          : f"USR_{i:05d}",
            "is_bot"           : 0,
            "account_age_days" : (start_date - created).days,
            "ip_address"       : fake.ipv4_public(),
            "email_domain"     : fake.free_email_domain(),
            "verified"         : random.choice([True, False]),
            "subscriber_count" : random.randint(0, 100000),
        })

    # Bot users
    for i in range(config["n_bot_users"]):
        created = start_date - timedelta(days=random.randint(0, 7))
        users.append({
            "user_id"          : f"BOT_{i:05d}",
            "is_bot"           : 1,
            "account_age_days" : (start_date - created).days,
            "ip_address"       : random.choice(bot_ip_pool),
            "email_domain"     : random.choice(["tempmail.com", "mailinator.com"]),
            "verified"         : False,
            "subscriber_count" : random.randint(0, 10),
        })

    return pd.DataFrame(users)

users_df = generate_users(CONFIG)
print(users_df["is_bot"].value_counts().rename({0: "Legitimate", 1: "Bot"}))
print(users_df.head(3))

is_bot
Legitimate    800
Bot           200
Name: count, dtype: int64
     user_id  is_bot  account_age_days       ip_address email_domain  \
0  USR_00000       0               684   210.72.237.234  hotmail.com   
1  USR_00001       0               311  215.213.125.185    gmail.com   
2  USR_00002       0               172    142.14.44.201  hotmail.com   

   verified  subscriber_count  
0      True              3278  
1      True             29256  
2      True             88696  


In [5]:
print("Legit user sample:")
print(users_df[users_df["is_bot"] == 0][["user_id","account_age_days","ip_address","email_domain","verified","subscriber_count"]].head(3))

print("\nBot user sample:")
print(users_df[users_df["is_bot"] == 1][["user_id","account_age_days","ip_address","email_domain","verified","subscriber_count"]].head(3))

Legit user sample:
     user_id  account_age_days       ip_address email_domain  verified  \
0  USR_00000               684   210.72.237.234  hotmail.com      True   
1  USR_00001               311  215.213.125.185    gmail.com      True   
2  USR_00002               172    142.14.44.201  hotmail.com      True   

   subscriber_count  
0              3278  
1             29256  
2             88696  

Bot user sample:
       user_id  account_age_days     ip_address    email_domain  verified  \
800  BOT_00000                 3  151.89.105.16    tempmail.com     False   
801  BOT_00001                 5   50.133.85.19  mailinator.com     False   
802  BOT_00002                 3  151.89.105.16  mailinator.com     False   

     subscriber_count  
800                 2  
801                 1  
802                 1  


In [6]:
# Generate event logs

def generate_events(users_df, config):
    events = []
    video_ids = [f"VID_{i:04d}" for i in range(config["n_videos"])]
    bot_target_videos = random.sample(video_ids, 20)
    sim_start = datetime(2024, 1, 1)

    for _, user in users_df.iterrows():
        is_bot = user["is_bot"] == 1

        for day_offset in range(config["days_window"]):
            day = sim_start + timedelta(days=day_offset)

            if is_bot:
                n_events = random.randint(*config["events_per_bot"])
                hours = [random.uniform(0, 24) for _ in range(n_events)]
                target_pool = bot_target_videos
                action_pool = ["like", "view", "view", "like"]
            else:
                n_events = random.randint(*config["events_per_legit"])
                hours = [random.triangular(7, 23, 14) for _ in range(n_events)]
                target_pool = video_ids
                action_pool = ["view", "like", "comment", "share", "flag"]

            for hour in hours:
                ts = day + timedelta(hours=hour)
                events.append({
                    "user_id"             : user["user_id"],
                    "is_bot"              : user["is_bot"],
                    "video_id"            : random.choice(target_pool),
                    "event_type"          : random.choice(action_pool),
                    "timestamp"           : ts,
                    "ip_address"          : user["ip_address"],
                    "session_duration_sec": random.randint(1, 10) if is_bot
                                           else random.randint(30, 3600),
                })

    return pd.DataFrame(events)

print("Generating events... this will take about 30 seconds")
events_df = generate_events(users_df, CONFIG)
print(f"Total events: {len(events_df):,}")
print(events_df.groupby(["is_bot", "event_type"]).size().unstack(fill_value=0))

Generating events... this will take about 30 seconds
Total events: 2,766,533
event_type  comment    flag     like   share     view
is_bot                                               
0            132681  132426   131408  131977   132539
1                 0       0  1052703       0  1052799


In [ ]:
users_df.to_csv("../data/users.csv", index=False)
events_df.to_csv("../data/events.csv", index=False)

print(f"users.csv  saved — {len(users_df):,} rows")
print(f"events.csv saved — {len(events_df):,} rows")


users.csv  saved — 1,000 rows
events.csv saved — 2,766,533 rows

Phase 1 complete ✓
Next → Phase 2: Feature Engineering
